In [ ]:
%pip install kagglehub
## ONLY RUN THIS IF YOU DON'T HAVE KAGGLEHUB INSTALLED ALREADY

In [2]:
from pathlib import Path
import csv

import pandas as pd

try:
    import kagglehub
except ImportError as error:
    raise ImportError(
        "Install kagglehub in this notebook first with: %pip install kagglehub"
    ) from error

# kagglehub returns a directory, so locate the CSV inside that directory.
download_directory = Path(
    kagglehub.dataset_download("sunnykakar/spotify-charts-all-audio-data")
)
csv_files = sorted(download_directory.rglob("*.csv"))
if len(csv_files) != 1:
    raise RuntimeError(
        f"Expected exactly one CSV in {download_directory}, found: {csv_files}"
    )
INPUT_PATH = csv_files[0]
OUTPUT_PATH = Path(r"data/spotify_charts_2019_2023.csv")

START_DATE = pd.Timestamp("2019-01-01")
END_DATE = pd.Timestamp("2023-12-31")
CHUNK_SIZE = 250_000

# The dataset is large; avoid loading the complete file into memory.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Downloaded dataset: {INPUT_PATH.resolve()}")
print(f"Output: {OUTPUT_PATH.resolve()}")
print(f"Date range: {START_DATE.date()} through {END_DATE.date()}")

/Users/amiebinan/Desktop/WPI/Y26-27/F26/MACHINE LEARNING/cs4342-song-performance/backend/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 2.84G/2.84G [00:53<00:00, 57.3MB/s]

Extracting files...


Downloaded dataset: /Users/amiebinan/.cache/kagglehub/datasets/sunnykakar/spotify-charts-all-audio-data/versions/1/merged_data.csv
Output: /Users/amiebinan/Desktop/WPI/Y26-27/F26/MACHINE LEARNING/cs4342-song-performance/data/spotify_charts_2019_2023.csv
Date range: 2019-01-01 through 2023-12-31


In [4]:
def read_column_names(path: Path) -> list[str]:
    """Read only the CSV header instead of loading the full dataset."""
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return next(csv.reader(handle))


if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {INPUT_PATH.resolve()}. "
        "Download the Kaggle CSV and update INPUT_PATH in the first cell."
    )

columns = read_column_names(INPUT_PATH)
print(f"Columns ({len(columns)}): {columns}")

date_columns = [
    column for column in columns
    if column.strip().lower() in {"date", "snapshot_date"}
]
if not date_columns:
    raise ValueError(
        "Could not find a chart-date column. Expected `date` or `snapshot_date`; "
        f"found: {columns}"
    )
DATE_COLUMN = date_columns[0]
print(f"Using date column: {DATE_COLUMN!r}")

Columns (29): ['', 'title', 'rank', 'date', 'artist', 'url', 'region', 'chart', 'trend', 'streams', 'track_id', 'album', 'popularity', 'duration_ms', 'explicit', 'release_date', 'available_markets', 'af_danceability', 'af_energy', 'af_key', 'af_loudness', 'af_mode', 'af_speechiness', 'af_acousticness', 'af_instrumentalness', 'af_liveness', 'af_valence', 'af_tempo', 'af_time_signature']
Using date column: 'date'


In [5]:
def clean_spotify_csv(
    input_path: Path,
    output_path: Path,
    date_column: str,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    chunksize: int = CHUNK_SIZE,
) -> dict[str, int]:
    """Filter a large Spotify CSV to an inclusive calendar-date range."""
    if start_date > end_date:
        raise ValueError("start_date must be on or before end_date")

    output_path.unlink(missing_ok=True)
    stats = {"rows_read": 0, "rows_kept": 0, "invalid_dates": 0, "chunks": 0}
    wrote_header = False

    for chunk in pd.read_csv(
        input_path,
        chunksize=chunksize,
        low_memory=False,
        encoding="utf-8-sig",
    ):
        stats["chunks"] += 1
        stats["rows_read"] += len(chunk)
        parsed_dates = pd.to_datetime(chunk[date_column], errors="coerce")
        stats["invalid_dates"] += int(parsed_dates.isna().sum())
        in_range = parsed_dates.between(start_date, end_date, inclusive="both")
        filtered = chunk.loc[in_range]
        stats["rows_kept"] += len(filtered)

        if not filtered.empty:
            filtered.to_csv(
                output_path,
                mode="a",
                index=False,
                header=not wrote_header,
                encoding="utf-8",
            )
            wrote_header = True

    if not wrote_header:
        raise ValueError("No rows matched the requested date range")
    return stats

In [6]:
stats = clean_spotify_csv(
    INPUT_PATH, OUTPUT_PATH, DATE_COLUMN, START_DATE, END_DATE
)
print(stats)
print(f"Wrote {stats['rows_kept']:,} rows to {OUTPUT_PATH.resolve()}")
print(f"Removed {stats['rows_read'] - stats['rows_kept']:,} rows outside the window")
print(f"Excluded {stats['invalid_dates']:,} rows with invalid or missing dates")

{'rows_read': 26174269, 'rows_kept': 16909377, 'invalid_dates': 0, 'chunks': 105}
Wrote 16,909,377 rows to /Users/amiebinan/Desktop/WPI/Y26-27/F26/MACHINE LEARNING/cs4342-song-performance/data/spotify_charts_2019_2023.csv
Removed 9,264,892 rows outside the window
Excluded 0 rows with invalid or missing dates


In [7]:
# Validate only the date column in chunks; the output file can be several GB.
import time

result_min = None
result_max = None
result_rows = 0
validation_started = time.perf_counter()

for chunk_number, chunk in enumerate(
    pd.read_csv(
        OUTPUT_PATH,
        usecols=[DATE_COLUMN],
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ),
    start=1,
):
    parsed_dates = pd.to_datetime(chunk[DATE_COLUMN], errors="coerce")
    if parsed_dates.isna().any():
        raise AssertionError("The cleaned file contains an invalid date")
    if not parsed_dates.between(START_DATE, END_DATE, inclusive="both").all():
        raise AssertionError("The cleaned file contains a date outside the requested range")

    result_rows += len(chunk)
    chunk_min = parsed_dates.min()
    chunk_max = parsed_dates.max()
    result_min = chunk_min if result_min is None else min(result_min, chunk_min)
    result_max = chunk_max if result_max is None else max(result_max, chunk_max)
    if chunk_number == 1 or chunk_number % 10 == 0:
        elapsed = time.perf_counter() - validation_started
        print(f"Validated {result_rows:,} rows ({elapsed:.1f}s elapsed)")

assert result_rows == stats["rows_kept"]
print(f"Validated {result_rows:,} rows")
print(f"Observed date range: {result_min.date()} through {result_max.date()}")
print(f"Output size: {OUTPUT_PATH.stat().st_size / (1024 ** 3):.2f} GiB")

Validated 250,000 rows (0.8s elapsed)
Validated 2,500,000 rows (7.9s elapsed)
Validated 5,000,000 rows (15.3s elapsed)
Validated 7,500,000 rows (22.5s elapsed)
Validated 10,000,000 rows (30.7s elapsed)
Validated 12,500,000 rows (38.6s elapsed)
Validated 15,000,000 rows (46.8s elapsed)
Validated 16,909,377 rows
Observed date range: 2019-01-01 through 2021-12-31
Output size: 17.45 GiB


In [8]:
# Make a smaller modeling copy while preserving market and track identity.
# Drop the source row index, track_id and URL; retain region for analysis/joins.
from collections import defaultdict
import time

CLEANED_PATH = Path(r"data/spotify_charts_2019_2023_cleaned.csv")
DROP_COLUMNS = {"", "track_id", "url", "available_markets"}
RENAME_COLUMNS = {"album": "album_name"}

if CLEANED_PATH.resolve() == OUTPUT_PATH.resolve():
    raise ValueError("Cleaned output must not overwrite the date-filtered source file")

CLEANED_PATH.unlink(missing_ok=True)
quality = {
    "rows_read": 0,
    "duplicate_rows_within_chunks": 0,
    "missing_values": defaultdict(int),
    "dtypes_by_column": defaultdict(set),
    "chunks": 0,
}
clean_min_date = None
clean_max_date = None
wrote_header = False
cleaning_started = time.perf_counter()

for chunk_number, chunk in enumerate(
    pd.read_csv(OUTPUT_PATH, chunksize=CHUNK_SIZE, low_memory=False),
    start=1,
):
    quality["chunks"] += 1
    quality["rows_read"] += len(chunk)

    remove_columns = [
        column for column in chunk.columns
        if (
            column.strip().lower() in DROP_COLUMNS
            or column.strip().lower().startswith("unnamed:")
        )
    ]
    chunk = chunk.drop(columns=remove_columns).rename(columns=RENAME_COLUMNS)

    quality["duplicate_rows_within_chunks"] += int(chunk.duplicated().sum())
    for column, count in chunk.isna().sum().items():
        quality["missing_values"][column] += int(count)
    for column, dtype in chunk.dtypes.items():
        quality["dtypes_by_column"][column].add(str(dtype))

    parsed_dates = pd.to_datetime(chunk[DATE_COLUMN], errors="coerce")
    chunk_min_date = parsed_dates.min()
    chunk_max_date = parsed_dates.max()
    clean_min_date = chunk_min_date if clean_min_date is None else min(clean_min_date, chunk_min_date)
    clean_max_date = chunk_max_date if clean_max_date is None else max(clean_max_date, chunk_max_date)

    chunk.to_csv(
        CLEANED_PATH,
        mode="a",
        index=False,
        header=not wrote_header,
        encoding="utf-8",
    )
    wrote_header = True

    if chunk_number == 1 or chunk_number % 10 == 0:
        elapsed = time.perf_counter() - cleaning_started
        print(f"Processed {quality['rows_read']:,} rows ({elapsed:.1f}s elapsed)")

if not wrote_header:
    raise ValueError("The date-filtered input file is empty")

print(f"Cleaned file: {CLEANED_PATH.resolve()}")
print(f"Rows written: {quality['rows_read']:,}")
print(f"Observed dates: {clean_min_date.date()} through {clean_max_date.date()}")
if clean_min_date > START_DATE or clean_max_date < END_DATE:
    print("WARNING: Source data does not cover the full configured five-year date range.")
print(f"Exact duplicate rows found within chunks: {quality['duplicate_rows_within_chunks']:,}")
print("Duplicate count is chunk-local; duplicates split across chunk boundaries are not counted.")
print("Missing values by column:")
print(pd.Series(dict(quality['missing_values'])).sort_values(ascending=False).to_string())
inconsistent_dtypes = {
    column: sorted(dtypes)
    for column, dtypes in quality["dtypes_by_column"].items()
    if len(dtypes) > 1
}
print("Inconsistent inferred dtypes across chunks:")
print(inconsistent_dtypes if inconsistent_dtypes else "None detected")
cleaned_size = CLEANED_PATH.stat().st_size
source_size = OUTPUT_PATH.stat().st_size
print(f"Cleaned CSV size: {cleaned_size / (1024 ** 3):.2f} GiB")
print(f"Size reduction: {(1 - cleaned_size / source_size) * 100:.1f}%")

Processed 250,000 rows (2.9s elapsed)
Processed 2,500,000 rows (28.6s elapsed)
Processed 5,000,000 rows (57.4s elapsed)
Processed 7,500,000 rows (85.4s elapsed)
Processed 10,000,000 rows (113.5s elapsed)
Processed 12,500,000 rows (142.8s elapsed)
Processed 15,000,000 rows (172.5s elapsed)
Cleaned file: /Users/amiebinan/Desktop/WPI/Y26-27/F26/MACHINE LEARNING/cs4342-song-performance/data/spotify_charts_2019_2023_cleaned.csv
Rows written: 16,909,377
Observed dates: 2019-01-01 through 2021-12-31
Exact duplicate rows found within chunks: 24
Duplicate count is chunk-local; duplicates split across chunk boundaries are not counted.
Missing values by column:
streams                3713307
album_name              258563
release_date            240293
af_energy               240293
af_tempo                240293
af_valence              240293
af_liveness             240293
af_instrumentalness     240293
af_acousticness         240293
af_speechiness          240293
af_mode                 240293


In [10]:
preview = pd.read_csv(CLEANED_PATH, nrows=1_000)
preview.to_csv("data/spotify_preview.csv", index=False)